# TIL 2026-11-18 — Vector Database와 pgvector (Lesson 06) 준비

## 오늘 한 일
- Lesson 06(PostgreSQL + pgvector) 실습을 시작하려고 환경을 점검했다.
- `psycopg`를 설치하고, 강의 코드를 `practice/my-ai-assistant/lesson06_pgvector.py`로 만들었다. (문법 검사 통과)
- 강의 본문(STEP 01~12)을 읽고, 이 PC에 적용했을 때 빠졌거나 어긋나는 부분을 정리했다.
- **DB 실습은 진행하지 못했다.** pgvector가 설치되어 있지 않아 STEP 02에서 HOLD.

## 환경 점검 결과
| 항목 | 결과 |
|---|---|
| PostgreSQL 18 서비스 | `localhost:5432` 실행 중 |
| pgvector 확장 파일 | 없음 (`share/extension`에 vector 파일 없음) |
| Visual Studio C++ 빌드 도구 | 없음 → 강의의 경로 B(소스 빌드) 불가 |
| `.env`의 `DB_*` 항목 | 없음 |
| `psycopg` | 설치됨 (3.3.9), 단 아래 DLL 문제 있음 |
| `.env` Git 제외 | `git check-ignore .env` 확인 |
| 전체 테스트 | 실패: 3개 모듈 import 오류 (`document_qa.py` 문법 오류) |

## STEP 12 기록표 (현재 기준)
| 항목 | 상태 | 근거 |
|---|---|---|
| pgvector 확장 | **HOLD** | 확장 파일 없음, 빌드 도구 없음 |
| Fixture 저장·재조회 | NOT_RUN | pgvector 미설치, `DB_*` 없음 |
| 중복 INSERT (Upsert) | NOT_RUN | 위와 같음 |
| Live 선택 실행 | NOT_RUN | 위와 같음 (API 호출도 하지 않음) |
| 근거 검토 | NOT_RUN | 위와 같음 |

## 겪은 문제
1. **`psycopg[binary]` DLL이 Windows 정책에 차단됨**
   - `ImportError: no pq wrapper available`, "애플리케이션 제어 정책에서 이 파일을 차단했습니다"
   - 우회 방법: PATH에 `C:\Program Files\PostgreSQL\18\bin`을 추가하면 PostgreSQL에 포함된 `libpq.dll`을 쓰는 순수 파이썬 모드로 import가 된다. 터미널을 열 때마다 필요하다. 정책 자체는 우회하지 않았다.
2. **`document_qa.py`가 어제부터 실행 불가**
   - 블로그 코드를 복사하면서 둥근 따옴표, `__` 밑줄, 들여쓰기가 깨짐
   - 이 파일을 import하는 `test_document_qa`, `test_embedding_search`, `test_vector_store`가 모두 실패한다.
   - 아직 고치지 않았다. (형식만 복구 / 원본으로 되돌리기 중 선택 필요)
3. `tests/embedding_search.py`가 `embedding_search.py`와 내용이 다른 채로 `tests/` 폴더에 들어 있다. (Git에 추가되지 않은 파일, 정리 필요)

## 배운 것
- **pgvector**는 PostgreSQL에 벡터 타입(`vector(n)`)과 거리 연산자를 추가하는 확장이다. 설치(`nmake install`)와 DB 안에서 활성화(`CREATE EXTENSION vector`)는 별개 단계다.
- **코사인 거리 `<=>`는 낮을수록 가깝다.** 거리 = 1 − 유사도. 같은 방향 0, 직각 1, 반대 방향 2. 거리는 정답 확률이 아니다.
- **문서 벡터는 한 번 저장해 재사용**하고, 질문 벡터는 검색할 때마다 새로 만든다. 그래서 `live-index`는 한 번이면 되지만 `live-query`는 호출할 때마다 API 비용이 생긴다.
- **`ON CONFLICT DO UPDATE`(upsert)** 는 같은 ID의 중복을 막지만, 삭제된 문단을 자동으로 정리하지는 않는다.
- **Fixture와 Live 테이블을 섞지 않는다.** 차원이 다르다(3 vs 1536).
- **문단이 3개뿐이면 `LIMIT 3`은 항상 전부 반환한다.** 무관한 질문도 결과가 나오므로, 결과가 있다는 것과 근거가 충분하다는 것은 별개다.

## 강의 글에서 발견한 빈틈
- DB(`ai_agent_lesson06`)를 만드는 방법이 없다. 본문은 "이미 생성됨"을 가정하므로 `CREATE DATABASE`를 먼저 해야 한다.
- 08-B가 `.env`를 "프로젝트 루트"에 두라고 하지만, 실제 `.env`는 `practice/my-ai-assistant/`에 있다.
- 본문의 `DBNAME`, `DBPASSWORD`, `OPENAIAPIKEY`는 밑줄이 사라진 표기다. (원래 `DB_NAME`, `DB_PASSWORD`, `OPENAI_API_KEY`) 코드 블록은 정상.
- `live-index` 없이 `live-query`를 실행하면 오류 없이 아무것도 출력하지 않는다.
- CLI가 모델·토큰 사용량을 출력하지 않아서 STEP 12의 "모델/차원/호출" 칸은 직접 적어야 한다.
- Lesson 05 글의 "다음 강의"는 Chroma였는데 Lesson 06은 pgvector다. 저장소의 `vector_store.py`, `test_vector_store.py`는 Chroma 기준이라 이번 강의와 별개다.

## 확인 문제 — 내 답
1. 코사인 유사도 0.9는 코사인 거리 얼마인가? → 0.1
2. 문서 벡터를 DB에 저장하면 무엇이 좋은가? → 변하지 않은 문서를 매번 임베딩하지 않아 계산과 비용이 줄어든다.
3. `ON CONFLICT DO UPDATE`가 하지 않는 일은? → 삭제된 문단을 자동으로 정리하지 않는다.
4. 환불 질문에 과제 제출 문단이 검색됐다면? → 검색은 됐지만 원문에 환불 정책이 없으면 근거 부족이다.

## 다음에 할 일
- [ ] pgvector 설치 방법 결정: Visual Studio C++ 빌드 도구를 설치해 경로 B로 갈지, HOLD로 두고 정리만 할지
- [ ] `document_qa.py` 복구 (형식만 복구 또는 `git restore`) 후 `python -m unittest discover -s tests -v`로 전체 테스트 확인
- [ ] `.env`에 `DB_HOST`, `DB_PORT`, `DB_NAME`, `DB_USER`, `DB_PASSWORD` 추가 (비밀번호는 채팅에 붙여 넣지 않기)
- [ ] 실습 DB 생성 후 `lesson06_pgvector.py check` → `index` → `count` → `query` 순서로 실행
- [ ] `tests/embedding_search.py` 정리
- [ ] 쌓인 변경(`main.py`, `model_info.py`, `.env.example`, `document_qa.py`, `lesson06_pgvector.py`, TIL 파일들) 정리 후 커밋
- [ ] Lesson 07 (Chunking과 검색 품질)

## 오늘의 한 줄
실행하지 못한 것은 PASS가 아니라 HOLD와 NOT_RUN으로 적는다. 막힌 이유(확장 파일 없음, 빌드 도구 없음)까지 남기면 다음에 바로 이어서 할 수 있다.
